In [2]:
"""
5. Lottery Ticket Hypothesis — ResNet-50 / CIFAR-10
====================================================
The Lottery Ticket Hypothesis (Frankle & Carlin, 2019):
  "A randomly initialized dense network contains a subnetwork
   (the winning ticket) that can be trained in isolation to match
   the accuracy of the original, in similar iterations."

Algorithm (Iterative Magnitude Pruning = IMP):
  1. Initialize network with random weights θ_0
  2. Train to convergence -> θ_T
  3. Prune p% of remaining weights (lowest magnitude) -> mask m
  4. Reset surviving weights to their INITIAL values: θ_0 ⊙ m
  5. Repeat from step 2 with the masked, re-initialized subnetwork

In our setting (pre-trained model, no fine-tuning loop):
  - We simulate the "winning ticket" by:
    a) Pruning at the final weights (θ_T = baseline)
    b) Masking to get the ticket structure
    c) Evaluating the masked model at final weights (not re-initialized)
  - We also test re-initialization (random θ_0 with ticket mask) to
    demonstrate the hypothesis difference.

Output: 5_lottery_ticket_Pruning.json
"""

import os, json, time, copy, tempfile, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

warnings.filterwarnings("ignore")

# ── Config ────────────────────────────────────────────────────────────────────
BASELINE_MODEL_PATH   = "../__2__baseline_resnet50_cifar10.pth"
BASELINE_METRICS_PATH = "../__2__baseline_metrics.json"
OUTPUT_JSON           = "5_v2_lottery_ticket_Pruning.json"

DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE  = 128
NUM_WORKERS = 2
NUM_CLASSES = 10

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

# Target FINAL sparsity levels (iterative steps will be computed per level)
SPARSITY_LEVELS   = [0.30] # , 0.50, 0.70, 0.80, 0.90
ITERATIVE_ROUNDS  = 5    # number of iterative pruning rounds to reach target
MAX_ACC_DROP      = 0.02
INFERENCE_RUNS    = 100


# ── Model ─────────────────────────────────────────────────────────────────────
def build_model(num_classes=10):
    model = models.resnet50(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, num_classes)
    return model

def load_baseline(path):
    model = build_model(NUM_CLASSES).to(DEVICE)
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    model.eval()
    return model

# ── Data ──────────────────────────────────────────────────────────────────────
def get_test_loader():
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = torchvision.datasets.CIFAR10(root="../data", train=False,
                                       download=True, transform=transform)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True)


# ── Lottery Ticket Helpers ────────────────────────────────────────────────────
def prunable_layers(model):
    return [(m, "weight") for _, m in model.named_modules()
            if isinstance(m, (nn.Conv2d, nn.Linear))]

def find_winning_ticket_mask(trained_model, target_sparsity, n_rounds):
    """
    Iterative magnitude pruning to find the ticket mask.
    Each round prunes per_round_rate of REMAINING weights.
    Returns: pruned model (trained weights × ticket mask)
    """
    # per_round rate so that n_rounds iterations reach target_sparsity
    # (1 - per_round)^n_rounds = (1 - target_sparsity)
    per_round = 1.0 - (1.0 - target_sparsity) ** (1.0 / n_rounds)

    current = copy.deepcopy(trained_model)

    for rnd in range(n_rounds):
        layers = prunable_layers(current)
        prune.global_unstructured(
            layers, pruning_method=prune.L1Unstructured, amount=per_round
        )
        for module, param in layers:
            prune.remove(module, param)

    return current


def extract_mask(model):
    """Return dict: layer_name -> binary mask tensor (1=survive, 0=pruned)."""
    mask = {}
    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            mask[name] = (module.weight.data != 0).float()
    return mask


def apply_mask_to_weights(target_model, mask_dict, use_random_init=False):
    """
    Apply a ticket mask to target_model.
    If use_random_init=True: reset surviving weights to kaiming normal (random init).
    If use_random_init=False: keep trained weights, zero out pruned positions.
    """
    result = copy.deepcopy(target_model)
    for name, module in result.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)) and name in mask_dict:
            m = mask_dict[name].to(DEVICE)
            if use_random_init:
                # Re-initialize survivors with kaiming normal (simulating θ_0)
                nn.init.kaiming_normal_(module.weight.data, mode='fan_out', nonlinearity='relu')
                module.weight.data *= m
            else:
                module.weight.data *= m
    return result


def real_sparsity(model):
    zeros = total = 0
    for _, m in model.named_modules():
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            zeros += (m.weight == 0).sum().item()
            total += m.weight.numel()
    return zeros / total if total else 0.0

def count_params(model):
    total  = sum(p.numel() for p in model.parameters())
    active = sum((p != 0).sum().item() for p in model.parameters())
    return total, active

def disk_size_mb(model):
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(model.state_dict(), tmp)
        size = os.path.getsize(tmp) / 1024**2
    finally:
        os.remove(tmp)
    return size

def sparse_size_mb(model):
    active = sum((p != 0).sum().item() for p in model.parameters())
    return active * 4 / 1024**2


# ── Evaluation ────────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    for inputs, lbls in loader:
        inputs = inputs.to(DEVICE, non_blocking=True)
        preds.extend(model(inputs).argmax(1).cpu().numpy())
        labels.extend(lbls.numpy())
    y_pred, y_true = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }

def measure_gpu_ms(model):
    if DEVICE.type != "cuda": return -1.0
    model.eval()
    dummy = torch.randn(BATCH_SIZE, 3, 32, 32, device=DEVICE)
    with torch.no_grad():
        for _ in range(20): model(dummy)
        torch.cuda.synchronize()
        s = torch.cuda.Event(enable_timing=True)
        e = torch.cuda.Event(enable_timing=True)
        s.record()
        for _ in range(INFERENCE_RUNS): model(dummy)
        e.record()
        torch.cuda.synchronize()
    return s.elapsed_time(e) / INFERENCE_RUNS

def measure_cpu_ms(model):
    cpu_m = copy.deepcopy(model).cpu().eval()
    dummy = torch.randn(BATCH_SIZE, 3, 32, 32)
    with torch.no_grad():
        for _ in range(5): cpu_m(dummy)
        t0 = time.perf_counter()
        for _ in range(20): cpu_m(dummy)
    return (time.perf_counter() - t0) / 20 * 1000

def compute_flops(model):
    """Total theoretical FLOPs — shape-based, no weight values needed."""
    flop_counts = {}
    hooks = []

    def make_hook(name):
        def hook(module, inp, out):
            if isinstance(module, nn.Conv2d):
                _, c_out, oH, oW = out.shape
                flop_counts[name] = 2 * module.in_channels * c_out * module.kernel_size[0] * module.kernel_size[1] * oH * oW
            elif isinstance(module, nn.Linear):
                flop_counts[name] = 2 * module.in_features * module.out_features
        return hook

    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            hooks.append(module.register_forward_hook(make_hook(name)))

    model.eval()
    with torch.no_grad():
        model(torch.randn(1, 3, 32, 32).to(DEVICE))

    for h in hooks: h.remove()
    return sum(flop_counts.values()), flop_counts

def compute_active_flops(model, flop_counts):
    """FLOPs scaled by per-layer weight density (approximates sparse compute)."""
    active = 0
    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)) and name in flop_counts:
            density = (module.weight != 0).float().mean().item()
            active += flop_counts[name] * density
    return int(active)

# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    print(f"\n{'='*62}")
    print("  5. Lottery Ticket Hypothesis — ResNet-50 / CIFAR-10")
    print(f"  Device: {DEVICE}  |  Rounds: {ITERATIVE_ROUNDS}")
    print(f"{'='*62}\n")

    with open(BASELINE_METRICS_PATH) as f:
        baseline = json.load(f)

    model  = load_baseline(BASELINE_MODEL_PATH)
    loader = get_test_loader()
    base_disk = disk_size_mb(model)
    base_flops, flop_counts = compute_flops(model)
    
    results    = []
    best_model = None
    best_score = -1.0
    best_sp    = None

    for sparsity in SPARSITY_LEVELS:
        print(f"\n  Target sparsity: {int(sparsity*100)}% | Rounds: {ITERATIVE_ROUNDS}")

        # Step 1: Find winning ticket via iterative magnitude pruning
        print(f"  Finding ticket mask (iterative, {ITERATIVE_ROUNDS} rounds)...")
        ticket_model = find_winning_ticket_mask(model, sparsity, ITERATIVE_ROUNDS)
        ticket_mask  = extract_mask(ticket_model)

        # Step 2a: Winning ticket — trained weights × mask (LTH claim: this should be good)
        print(f"  Evaluating: trained weights × ticket mask...")
        winning_ticket = apply_mask_to_weights(model, ticket_mask, use_random_init=False)
        actual_sp      = real_sparsity(winning_ticket)
        total_p, active_p = count_params(winning_ticket)
        metrics_ticket = evaluate(winning_ticket, loader)
        inf_gpu        = measure_gpu_ms(winning_ticket)
        inf_cpu        = measure_cpu_ms(winning_ticket)
        acc_drop_ticket = baseline["accuracy"] - metrics_ticket["accuracy"]
        sp_mb  = sparse_size_mb(winning_ticket)
        dk_mb  = disk_size_mb(winning_ticket)

        # Step 2b: Random ticket — random init weights × same mask (control: should be worse)
        print(f"  Evaluating: random init × same mask (control)...")
        random_ticket  = apply_mask_to_weights(model, ticket_mask, use_random_init=True)
        metrics_random = evaluate(random_ticket, loader)
        acc_drop_random = baseline["accuracy"] - metrics_random["accuracy"]

        print(f"  Ticket acc: {metrics_ticket['accuracy']:.4f} (drop: {acc_drop_ticket:+.4f}) | "
              f"Random acc: {metrics_random['accuracy']:.4f} (drop: {acc_drop_random:+.4f})")
        print(f"  -> LTH advantage: {(metrics_ticket['accuracy'] - metrics_random['accuracy'])*100:.2f}%")

        row = {
            "target_sparsity"                : sparsity,
            "actual_sparsity"                : round(actual_sp, 4),
            "iterative_rounds"               : ITERATIVE_ROUNDS,
            # Winning ticket (trained weights)
            "winning_ticket_accuracy"        : round(metrics_ticket["accuracy"], 6),
            "winning_ticket_precision"       : round(metrics_ticket["precision"], 6),
            "winning_ticket_recall"          : round(metrics_ticket["recall"], 6),
            "winning_ticket_f1"              : round(metrics_ticket["f1"], 6),
            "winning_ticket_accuracy_drop"   : round(acc_drop_ticket, 6),
            # Random init control
            "random_init_accuracy"           : round(metrics_random["accuracy"], 6),
            "random_init_accuracy_drop"      : round(acc_drop_random, 6),
            # LTH advantage
            "lth_advantage_pct"              : round((metrics_ticket["accuracy"] - metrics_random["accuracy"]) * 100, 4),
            # Shared
            "params_total"                   : total_p,
            "params_active"                  : active_p,
            "size_sparse_theoretical_mb"     : round(sp_mb, 4),
            "size_disk_mb"                   : round(dk_mb, 4),
            "disk_saved_mb"                  : round(base_disk - dk_mb, 4),
            "inference_gpu_ms"               : round(inf_gpu, 4) if inf_gpu > 0 else None,
            "inference_cpu_ms"               : round(inf_cpu, 4),
            "flops_total"   : int(base_flops),
            "flops_active"  : compute_active_flops(winning_ticket, flop_counts),
            "flops_reduction_pct": round((1 - compute_active_flops(winning_ticket, flop_counts) / base_flops) * 100, 2)
        }
        results.append(row)

        if acc_drop_ticket <= MAX_ACC_DROP:
            score = metrics_ticket["accuracy"] + ((base_disk - dk_mb) / base_disk) * 0.1
            if score > best_score:
                best_score, best_model, best_sp = score, copy.deepcopy(winning_ticket), sparsity

    report = {
        "method"              : "lottery_ticket_hypothesis",
        "description"         : "LTH: iterative magnitude pruning to find ticket mask; compare trained-weights vs random-init",
        "pruning_granularity" : "weight (unstructured)",
        "iterative_rounds"    : ITERATIVE_ROUNDS,
        "baseline"            : baseline,
        "device"              : str(DEVICE),
        "max_acc_drop_threshold": MAX_ACC_DROP,
        "best_sparsity"       : best_sp,
        "results"             : results,
        "notes": {
            "winning_ticket"    : "Ticket mask + trained (θ_T) weights. Represents pruned baseline.",
            "random_init"       : "Same mask + kaiming-normal re-initialized weights. Control: untrained subnetwork.",
            "lth_advantage_pct" : "Positive = trained weights preserve accuracy vs random init. Validates LTH.",
            "full_lth"          : "True LTH trains the re-initialized subnetwork; here we only evaluate it untrained as a demonstration.",
            "reference"         : "Frankle & Carlin (2019) https://arxiv.org/abs/1803.03635",
        }
    }

    with open(OUTPUT_JSON, "w") as f:
        json.dump(report, f, indent=2)
    print(f"\n  ✓ Saved -> {OUTPUT_JSON}")


if __name__ == "__main__":
    main()



  5. Lottery Ticket Hypothesis — ResNet-50 / CIFAR-10
  Device: cuda  |  Rounds: 5


  Target sparsity: 30% | Rounds: 5
  Finding ticket mask (iterative, 5 rounds)...
  Evaluating: trained weights × ticket mask...
  Evaluating: random init × same mask (control)...
  Ticket acc: 0.9321 (drop: -0.0001) | Random acc: 0.1000 (drop: +0.8320)
  -> LTH advantage: 83.21%

  ✓ Saved -> 5_v2_lottery_ticket_Pruning.json
